In [1]:
from langchain.retrievers import EnsembleRetriever, BM25Retriever
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

/Users/spender/Coding Folder/Rag/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/spender/Coding Folder/Rag/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
%pip install -q rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ──────────────────────────────────────────────────────────────────
# SETUP: Create our sample company data
# ──────────────────────────────────────────────────────────────────

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.",
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.",
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
    "Orange County reported new housing developments."
]

In [4]:
# Convert to Document objects for LangChain
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

print("Sample Data:")
for i, chunk in enumerate(chunks, 1):
    print(f"{i}. {chunk}")

print("\n" + "="*80)

Sample Data:
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java pr

# 1. Vector Retriever (Semantic Search / Dense Retrieval)

In [5]:
print("Setting up Vector Retriever...")
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Vector Retriever ready!")

Setting up Vector Retriever...
Vector Retriever ready!


In [6]:
# Test semantic search (works conceptually even without exact keywords)
test_query = "space exploration company"

print(f"Testing Vector Search with: '{test_query}'")
test_docs = vector_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing Vector Search with: 'space exploration company'
Found: SpaceX develops Starship rockets for Mars missions.
Found: Cybertruck reservations exceeded company expectations.


# 2. BM25 Retriever (Keyword Search / Sparse Retrieval)

In [7]:
print("Setting up BM25 Retriever...")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3
print("BM25 Retriever ready!")

Setting up BM25 Retriever...
BM25 Retriever ready!


In [8]:
# Test exact keyword matching
test_query = "Cybertruck"

print(f"Testing BM25 Search with: '{test_query}'")
test_docs = bm25_retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing BM25 Search with: 'Cybertruck'
Found: Cybertruck reservations exceeded company expectations.
Found: Tesla Cybertruck production ramp begins in 2024.
Found: Orange juice sales increased during winter months.


# 3. Hybrid Retriever (Ensemble of Vector + BM25)

In [9]:
print("Setting up Hybrid Retriever...")
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]  # 70% Semantic, 30% Keyword matching
)
print("Hybrid Retriever setup complete!\n")

Setting up Hybrid Retriever...
Hybrid Retriever setup complete!



In [10]:
# Query 1: Mixed semantic and exact numbers
# Vector search understands "purchase cost" semantically
# BM25 search finds exact "7.5 billion"
test_query = "purchase cost 7.5 billion"
print(f"Query 1: '{test_query}'")

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print("\nQuery 1 demonstrates finding exact financial figures using both semantic and keyword matching.")

Query 1: 'purchase cost 7.5 billion'
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Cybertruck reservations exceeded company expectations.
3. Orange juice sales increased during winter months.
4. Tesla Cybertruck production ramp begins in 2024.

Query 1 demonstrates finding exact financial figures using both semantic and keyword matching.


In [11]:
# Query 2: Semantic concept + specific product name
# Vector search understands "electric vehicle manufacturing"
# BM25 search finds exact "Cybertruck"
test_query = "electric vehicle manufacturing Cybertruck"
print(f"Query 2: '{test_query}'")

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print("\nQuery 2 demonstrates combining product-specific keywords with broader conceptual queries.")

Query 2: 'electric vehicle manufacturing Cybertruck'
1. Tesla Cybertruck production ramp begins in 2024.
2. Cybertruck reservations exceeded company expectations.
3. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.

Query 2 demonstrates combining product-specific keywords with broader conceptual queries.


In [12]:
# Query 3: Where neither alone would be perfect
# "Company performance" is semantic, "Tesla" is exact keyword
test_query = "company performance Tesla"
print(f"Query 3: '{test_query}'")

retrieved_chunks = hybrid_retriever.invoke(test_query)
for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc.page_content}")
print("\nQuery 3 shows how hybrid handles mixed queries better than either approach alone.")

Query 3: 'company performance Tesla'
1. Tesla Tesla Tesla financial quarterly results improved significantly.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Cybertruck reservations exceeded company expectations.

Query 3 shows how hybrid handles mixed queries better than either approach alone.


In [13]:
# Generate final answer with LLM using hybrid context
combined_input = f"""Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f'- {doc.page_content}' for doc in retrieved_chunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # or gpt-4o

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

print("Generating answer with LLM...")
result = model.invoke(messages)

print("\n--- Generated Response ---")
print("Content only:")
print(result.content)

Generating answer with LLM...

--- Generated Response ---
Content only:
Based on the provided documents, Tesla's performance appears to be strong. The company reported significant improvements in its financial quarterly results and continues to lead in the electric vehicle market. Additionally, Tesla announced new manufacturing facilities, which suggests growth and expansion. Furthermore, the reservations for the Cybertruck have exceeded company expectations, indicating positive consumer interest and demand. Overall, these points suggest that Tesla is performing well.
